# Documentation and resources

**Big Query**
- Colab has an example notebook on BigQuery too.  File > Open notebook > Examples > Getting Started with BigQuery.
- Also a Big Query Snippets Example Notebook
- [BigQuery Documentation]( https://cloud.google.com/bigquery/docs )
- [Open Data Sets]( https://console.cloud.google.com/marketplace/browse?filter=solution-type:dataset )
- [Reddit - list of data sets available on BQ]( https://www.reddit.com/r/bigquery/wiki/datasets )

**Big Query Console**  
- [Google Cloud Console]( https://console.cloud.google.com )
- Make sure your project is selected
- Scroll down to BigQuery on the left menu
- [Setup and query instructions]( https://cloud.google.com/bigquery/docs/quickstarts/query-public-dataset-console )

**SQL**
- [Kaggle Intro to SQL]( https://www.kaggle.com/learn/intro-to-sql ) uses BigQuery
- [Kaggle Advanced SQL]( https://www.kaggle.com/learn/advanced-sql )

**BigQuery Public Data Sets**
- [Austin Crime]( https://console.cloud.google.com/marketplace/product/city-of-austin/austin-crime )


#  Linking BigQuery to Colab

## Getting started

**You will only need to do this part once.**

1. Use the [Cloud Resource Manager](https://console.cloud.google.com/cloud-resource-manager) to Create a Cloud Platform project if you do not already have one.

    - Create Project
    - Project Name

2. [Enable BigQuery APIs](https://console.cloud.google.com/flows/enableapi?apiid=bigquery) for the project

**Note:** You get 1 TB/month of free queries for open datasets
- Kaggle gives you 5 TB/month free



## Imports

In [1]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
from google.colab import auth
from google.colab import syntax
from google.colab import userdata


In [2]:
%alias gcloud gcloud

### Provide your credentials

In [3]:
auth.authenticate_user()
print('Authenticated')

Authenticated


## Optional: Enable data table display

Colab includes the ``google.colab.data_table`` package that can be used to display large pandas dataframes as an interactive data table.
It can be enabled with:

In [4]:
%load_ext google.colab.data_table
%unload_ext google.colab.data_table

If you would prefer to return to the classic Pandas dataframe display, you can disable this by running:

In [5]:
%unload_ext google.colab.data_table

The google.colab.data_table extension is not loaded.


## List projects



In order to query BigQuery, you will need to specify a project ID.  To get a list of project IDs associated with your account, run the following command.

In [6]:
gcloud projects list --sort-by=projectId

PROJECT_ID                      NAME              PROJECT_NUMBER  ENVIRONMENT
my-ddds-project                 My-DDDS-Project   62420001175
project-9e8020bd-e910-4e44-ab9  My First Project  510894428594
project-a7c86939-191d-4cdb-bcd  My First Project  23897434908


To take a quick anonymous survey, run:
  $ gcloud survey



## Declare the Cloud project ID which will be used throughout this notebook

Create a secret in your vault named `bq_billing_project_id` and assign it the value of the project ID you want to use.

1. open the vault ( key icon on the far left )
1. click "Add new secret"
1. type the name ( `bq_billing_project_id` )
1. paste the value ( do NOT type enter )
1. click back in the notebook


Fetch the project ID from your vault.

In [7]:
project_id = userdata.get('bq_billing_project_id')
project_id

'my-ddds-project'

## Samples data set



The [GSOD table](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=samples&t=gsod&page=table) in the Samples data set contains weather information collected by NOAA, such as precipitation amounts and wind speeds from late 1929 to early 2010.


# Use BigQuery via magics



The `google.cloud.bigquery` library also includes a magic command which runs a query and either displays the result or saves it to a variable as a `DataFrame`.

In [8]:
# Display query output immediately

%%bigquery --project {project_id}
SELECT
  COUNT(1) as total_rows
FROM `bigquery-public-data.samples.gsod`


Query is running:   0%|          |

Downloading:   0%|          |

,total_rows
0,114420316


In [9]:
# Save output in a variable `df`

%%bigquery bq_public_data_df --project {project_id}
SELECT
  COUNT(1) as total_rows
FROM `bigquery-public-data.samples.gsod`


Query is running:   0%|          |

Downloading:   0%|          |

In [10]:
bq_public_data_df

,total_rows
0,114420316


In [11]:
f'{bq_public_data_df.iloc[0,0]:,.0f}'

'114,420,316'

# Use BigQuery through google-cloud-bigquery



See [BigQuery documentation](https://cloud.google.com/bigquery/docs) and [library reference documentation](https://googlecloudplatform.github.io/google-cloud-python/latest/bigquery/usage.html).


### Count total number of rows

In [12]:
query = '''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.samples.gsod`
  '''

with bigquery.Client(project=project_id) as client:
  bq_df = client.query( query ).to_dataframe()


In [13]:
bq_df.shape


(1, 1)

In [14]:
bq_df

,total
0,114420316


In [15]:
query = '''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.samples.gsod`
  '''

with bigquery.Client(project=project_id) as client:
  row_count = client.query( query ).to_dataframe()["total"][0]

print(f'Full dataset has {row_count:,.0f} rows')


Full dataset has 114,420,316 rows


## Sample approximately 2000 random rows

### Describe the sampled data

In [16]:
sample_count = 2000

query = f'''
  SELECT
    *
  FROM
    `bigquery-public-data.samples.gsod`
  WHERE RAND() < {sample_count}/{row_count}
'''

with bigquery.Client(project=project_id) as client:
  df = client.query( query ).to_dataframe()

df


,station_number,wban_number,year,month,day,mean_temp,num_mean_temp_samples,mean_dew_point,num_mean_dew_point_samples,mean_sealevel_pressure,...,min_temperature,min_temperature_explicit,total_precipitation,snow_depth,fog,rain,snow,hail,thunder,tornado
0,722650,23023,1960,8,26,83.900002,24,64.900002,24,1010.900024,...,NaN,<NA>,NaN,NaN,False,False,False,False,False,False
1,679830,99999,1962,8,30,66.599998,5,35.799999,5,NaN,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
2,319810,99999,1973,5,1,46.400002,8,34.700001,8,1016.299988,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
3,726930,24221,1973,9,17,63.400002,24,52.299999,24,1016.099976,...,NaN,<NA>,NaN,NaN,False,False,False,False,False,False
4,836810,99999,1975,10,27,71.900002,11,54.099998,10,1016.900024,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1943,947520,99999,1998,1,28,78.300003,8,51.799999,8,1006.599976,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
1944,725098,99999,2004,7,17,73.900002,24,63.200001,24,1008.500000,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
1945,723417,93988,2006,4,18,79.099998,24,66.000000,24,1006.000000,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
1946,768056,99999,2008,3,30,79.400002,24,69.900002,24,1011.799988,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False


In [17]:
df.index.size

1948

In [18]:
df.describe().transpose().astype({"count": int})

,count,mean,std,min,25%,50%,75%,max
station_number,1948,501532.034908,302077.174578,10080.0,232740.0,526810.0,725122.0,999999.0
wban_number,1948,90654.375257,25798.774008,1.0,99999.0,99999.0,99999.0,99999.0
year,1948,1986.822895,16.161675,1933.0,1977.0,1989.0,2001.0,2010.0
month,1948,6.460472,3.423532,1.0,3.0,7.0,9.0,12.0
day,1948,15.675051,8.80489,1.0,8.0,16.0,23.0,31.0
mean_temp,1948,52.175051,24.163029,-74.900002,37.900002,55.35,71.199997,100.099998
num_mean_temp_samples,1948,12.810575,7.874109,4.0,7.0,8.0,23.0,24.0
mean_dew_point,1861,41.878775,22.289668,-83.099998,29.0,44.099998,57.700001,82.300003
num_mean_dew_point_samples,1861,12.759269,7.888748,4.0,7.0,8.0,23.0,24.0
mean_sealevel_pressure,1483,1014.798785,9.291445,948.700012,1009.049988,1014.299988,1020.099976,1061.400024


### View the first 10 rows

In [19]:
df.head(10)

,station_number,wban_number,year,month,day,mean_temp,num_mean_temp_samples,mean_dew_point,num_mean_dew_point_samples,mean_sealevel_pressure,...,min_temperature,min_temperature_explicit,total_precipitation,snow_depth,fog,rain,snow,hail,thunder,tornado
0,722650,23023,1960,8,26,83.900002,24,64.900002,24,1010.900024,...,NaN,<NA>,NaN,NaN,False,False,False,False,False,False
1,679830,99999,1962,8,30,66.599998,5,35.799999,5,NaN,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
2,319810,99999,1973,5,1,46.400002,8,34.700001,8,1016.299988,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
3,726930,24221,1973,9,17,63.400002,24,52.299999,24,1016.099976,...,NaN,<NA>,NaN,NaN,False,False,False,False,False,False
4,836810,99999,1975,10,27,71.900002,11,54.099998,10,1016.900024,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
5,972300,99999,1976,6,28,79.199997,5,72.000000,5,1011.000000,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
6,467340,99999,1979,2,23,75.699997,23,71.699997,23,1008.799988,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
7,401530,99999,1980,3,23,62.200001,4,39.200001,4,NaN,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False
8,702460,26512,1988,11,18,-1.100000,7,-6.400000,7,NaN,...,NaN,<NA>,NaN,NaN,False,False,False,False,False,False
9,718510,99999,1996,7,25,65.500000,19,54.099998,19,1019.900024,...,NaN,<NA>,0.0,NaN,False,False,False,False,False,False


In [20]:
df.isnull().sum()

,0
station_number,0
wban_number,0
year,0
month,0
day,0
mean_temp,0
num_mean_temp_samples,0
mean_dew_point,87
num_mean_dew_point_samples,87
mean_sealevel_pressure,465


In [21]:
# 10 highest total_precipitation samples
(
df
  .sort_values('total_precipitation', ascending=False)
  .head(10)
  [['station_number', 'year', 'month', 'day', 'total_precipitation']]
)


,station_number,year,month,day,total_precipitation
89,942910,1992,12,18,9.53
293,591020,2000,8,25,6.97
1248,726440,2008,6,12,3.40
195,303930,1962,1,26,2.95
1393,851410,1993,3,3,2.91
524,484260,1983,9,30,2.56
640,474300,1988,6,9,2.48
1545,722420,1977,11,29,2.40
1920,679830,1998,3,19,2.17
1726,725054,2003,5,27,2.15


# Use BigQuery through pandas-gbq



The `pandas-gbq` library is a community led project by the pandas community. It covers basic functionality, such as writing a DataFrame to BigQuery and running a query, but as a third-party library it may not handle all BigQuery features or use cases.

[Pandas GBQ Documentation]( https://pandas-gbq.readthedocs.io/en/latest/ )

In [22]:
query = '''
  SELECT
    name, SUM(number) as count
  FROM
    bigquery-public-data.usa_names.usa_1910_2013
  WHERE
    state = 'TX'
  GROUP BY
    name
  ORDER BY
    count DESC
  LIMIT
    100
'''

df = pandas_gbq.read_gbq( query, project_id=project_id, )
df.head()


Downloading: 100%|██████████|


,name,count
0,James,272793
1,John,235139
2,Michael,225320
3,Robert,220399
4,David,219028
